# IMC Prosperity 4 — Round 5: Optimized Basket Cointegration Analysis

**Objetivo:** Evolucionar el modelo de regresión simple (Equal-Weighted Basket) a un modelo de regresión múltiple y análisis de autovectores de Johansen. Esto nos permitirá encontrar cestas estacionarias ($1$ vs $N$) cuando las relaciones de pares puros ($1$ vs $1$) colapsan debido a ruido idiosincrático.

Este notebook integra de forma autónoma la carga de datos, cálculo de p-valores (ADF) y *Half-life* para validar directamente los portafolios creados.

In [3]:
# ─── CELL 1: Imports & Configuration ────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint, ccf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from numpy.lib.stride_tricks import sliding_window_view
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 9})
sns.set_style('darkgrid')

DATA_DIR = r'C:\Users\Usuario\Desktop\prosperity-4\Round5\data'
DAYS = [2, 3, 4]
POS_LIMIT = 10
ROUND_NUM = 5

CATEGORIES = {
    'Galaxy Sounds':  ['GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_BLACK_HOLES',
                       'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_WINDS',
                       'GALAXY_SOUNDS_SOLAR_FLAMES'],
    'Sleep Pods':     ['SLEEP_POD_SUEDE', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_POLYESTER',
                       'SLEEP_POD_NYLON', 'SLEEP_POD_COTTON'],
    'Microchips':     ['MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_SQUARE',
                       'MICROCHIP_RECTANGLE', 'MICROCHIP_TRIANGLE'],
    'Pebbles':        ['PEBBLES_XS', 'PEBBLES_S', 'PEBBLES_M', 'PEBBLES_L', 'PEBBLES_XL'],
    'Robots':         ['ROBOT_VACUUMING', 'ROBOT_MOPPING', 'ROBOT_DISHES',
                       'ROBOT_LAUNDRY', 'ROBOT_IRONING'],
    'UV Visors':      ['UV_VISOR_YELLOW', 'UV_VISOR_AMBER', 'UV_VISOR_ORANGE',
                       'UV_VISOR_RED', 'UV_VISOR_MAGENTA'],
    'Translators':    ['TRANSLATOR_SPACE_GRAY', 'TRANSLATOR_ASTRO_BLACK',
                       'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST',
                       'TRANSLATOR_VOID_BLUE'],
    'Panels':         ['PANEL_1X2', 'PANEL_2X2', 'PANEL_1X4', 'PANEL_2X4', 'PANEL_4X4'],
    'Shakes':         ['OXYGEN_SHAKE_MORNING_BREATH', 'OXYGEN_SHAKE_EVENING_BREATH',
                       'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_GARLIC'],
    'Snacks':         ['SNACKPACK_CHOCOLATE', 'SNACKPACK_VANILLA', 'SNACKPACK_PISTACHIO',
                       'SNACKPACK_STRAWBERRY', 'SNACKPACK_RASPBERRY'],
}

print('Categories defined. Total products:', sum(len(v) for v in CATEGORIES.values()))

Categories defined. Total products: 50


In [4]:
# ─── CELL 2: Data Ingestion & Cleaning ──────────────────────────────────────

def load_prices(data_dir: str, days: list) -> pd.DataFrame:
    """
    Load all price CSVs, concatenate, and pivot to wide format.
    Returns: DataFrame indexed by (day * 1_000_000 + timestamp) with one
             column per product containing mid_price.
    """
    frames = []
    for day in days:
        path = os.path.join(data_dir, f'prices_round_{ROUND_NUM}_day_{day}.csv')
        # Try to read, pass if file is missing for smooth error handling locally
        if not os.path.exists(path):
            print(f"Warning: File {path} not found. Skipping day {day}.")
            continue
        df = pd.read_csv(path, sep=';')
        df['global_ts'] = day * 1_000_000 + df['timestamp']
        frames.append(df[['global_ts', 'day', 'timestamp', 'product', 'mid_price',
                           'bid_price_1', 'ask_price_1']])

    if not frames:
        raise ValueError(f"No valid CSV data found in {data_dir} for days {days}")

    raw = pd.concat(frames, ignore_index=True)
    raw = raw.sort_values(['product', 'global_ts'])

    # Pivot: rows = global_ts, columns = product, values = mid_price
    pivot = raw.pivot_table(index='global_ts', columns='product', values='mid_price')
    pivot.columns.name = None
    pivot = pivot.sort_index()

    bid_pivot = raw.pivot_table(index='global_ts', columns='product', values='bid_price_1')
    bid_pivot.columns.name = None
    ask_pivot = raw.pivot_table(index='global_ts', columns='product', values='ask_price_1')
    ask_pivot.columns.name = None

    print(f'Loaded {len(pivot):,} timestamps × {pivot.shape[1]} products')
    return pivot, bid_pivot, ask_pivot, raw

# Ejecutar carga (requiere que la ruta en DATA_DIR sea correcta y accesible)
prices, bids, asks, raw_df = load_prices(DATA_DIR, DAYS)


Loaded 30,000 timestamps × 50 products


In [5]:
# ─── CELL 3: NumPy Helpers (ADF, Half-Life, Hurst) ──────────────────────────

def _simple_ols_alpha_beta(y, x):
    """Closed-form OLS for y = α + β·x."""
    y = np.asarray(y, dtype=float).ravel()
    x = np.asarray(x, dtype=float).ravel()
    x_mean = x.mean()
    y_mean = y.mean()
    x_cent = x - x_mean
    denom = np.dot(x_cent, x_cent)
    if denom <= 1e-12:
        return np.nan, np.nan
    beta = float(np.dot(x_cent, y - y_mean) / denom)
    alpha = float(y_mean - beta * x_mean)
    return alpha, beta

def adf_test(series):
    """ADF unit-root test (statsmodels.adfuller)."""
    s = np.asarray(series).ravel()
    s = s[~np.isnan(s)]
    if len(s) < 30:
        return {'adf_stat': np.nan, 'adf_pval': np.nan, 'stationary': False}
    res = adfuller(s, maxlag=20, autolag='AIC', regression='c')
    return {'adf_stat': res[0], 'adf_pval': res[1], 'stationary': res[1] < 0.05}

def half_life(spread):
    """
    Mean-reversion half-life from AR(1):  Δs_t = α + β·s_{t−1}.
    Returns ∞ if non-stationary or β ≥ 0.
    """
    s = np.asarray(spread).ravel()
    s = s[~np.isnan(s)]
    if len(s) < 50:
        return np.inf
    delta = np.diff(s)
    lag = s[:-1]
    _, beta = _simple_ols_alpha_beta(delta, lag)
    return float(-np.log(2) / beta) if np.isfinite(beta) and beta < 0 else np.inf

def hurst_exponent(series, max_lag: int = 100) -> float:
    """Vectorized Hurst estimate. H<0.5 mean-reverting, H≈0.5 random walk."""
    s = np.asarray(series, dtype=float).ravel()
    s = s[~np.isnan(s)]
    if len(s) < 200:
        return np.nan
    lags = np.arange(2, min(max_lag, len(s) // 4))
    if len(lags) < 2:
        return np.nan
    width = len(s) - lags[0]
    offsets = np.arange(width)
    valid_offsets = offsets[None, :] < (len(s) - lags[:, None])
    lagged_idx = np.minimum(lags[:, None] + offsets[None, :], len(s) - 1)
    diffs = s[lagged_idx] - s[offsets][None, :]
    diffs = np.where(valid_offsets, diffs, np.nan)
    tau = np.nanstd(diffs, axis=1)
    valid = tau > 0
    if valid.sum() < 2:
        return np.nan
    return float(np.polyfit(np.log(lags[valid]), np.log(tau[valid]), 1)[0])

print('NumPy Helpers cargados correctamente.')


NumPy Helpers cargados correctamente.


## 1. Múltiple OLS para Cobertura de Cestas
Asumimos la relación: $$Y_t = \alpha + \beta_1 X_{1,t} + \beta_2 X_{2,t} + \beta_3 X_{3,t} + \beta_4 X_{4,t} + S_t$$
El residuo $S_t$ es el *spread* que operaremos mediante reversión a la media.

In [6]:
basket_ols_results = []

for cat, cols in CATEGORIES.items():
    cat_df = prices[cols].dropna()
    
    for target in cols:
        others = [c for c in cols if c != target]
        
        Y = cat_df[target]
        X = cat_df[others]
        X_const = sm.add_constant(X) # Intercepto alpha
        
        # Ajuste Múltiple OLS
        model = sm.OLS(Y, X_const).fit()
        spread = model.resid
        
        # Diagnósticos de estacionariedad
        adf_res = adf_test(spread)
        hl = half_life(spread)
        hu = hurst_exponent(spread)
        
        # Extraer betas (pesos de cobertura para la cesta)
        betas = model.params.drop('const')
        
        basket_ols_results.append({
            'category': cat,
            'target': target,
            'adf_pval': round(adf_res['adf_pval'], 5),
            'half_life': round(hl, 1) if np.isfinite(hl) else 9999,
            'hurst': round(hu, 4) if not np.isnan(hu) else np.nan,
            'r_squared': round(model.rsquared, 4),
            'b1': round(betas.iloc[0], 3), 
            'b2': round(betas.iloc[1], 3),
            'b3': round(betas.iloc[2], 3),
            'b4': round(betas.iloc[3], 3)
        })

opt_basket_df = pd.DataFrame(basket_ols_results).sort_values('adf_pval')
print('=== 1 Asset vs. 4-Asset Optimized Basket (Multiple OLS) ===')
print(f'Stationary at 5%: {(opt_basket_df["adf_pval"] < 0.05).sum()}')
display(opt_basket_df.head(20))

=== 1 Asset vs. 4-Asset Optimized Basket (Multiple OLS) ===
Stationary at 5%: 37


,category,target,adf_pval,half_life,hurst,r_squared,b1,b2,b3,b4
15,Pebbles,PEBBLES_XS,0.00000,0.7,0.0002,1.0000,-1.000,-1.000,-1.000,-1.000
18,Pebbles,PEBBLES_L,0.00000,0.7,0.0002,1.0000,-1.000,-1.000,-1.000,-1.000
17,Pebbles,PEBBLES_M,0.00000,0.7,0.0002,1.0000,-1.000,-1.000,-1.000,-1.000
16,Pebbles,PEBBLES_S,0.00000,0.7,0.0002,1.0000,-1.000,-1.000,-1.000,-1.000
19,Pebbles,PEBBLES_XL,0.00000,0.7,0.0002,1.0000,-1.000,-1.000,-1.000,-1.000
14,Microchips,MICROCHIP_TRIANGLE,0.00012,488.3,0.4956,0.8587,-0.177,0.564,0.031,-0.299
11,Microchips,MICROCHIP_OVAL,0.00047,554.4,0.4904,0.9186,-0.329,-0.233,0.455,1.126
29,UV Visors,UV_VISOR_MAGENTA,0.00076,586.3,0.4740,0.8147,0.031,-0.657,-0.065,-0.287
26,UV Visors,UV_VISOR_AMBER,0.00113,588.3,0.4769,0.8964,-0.167,-0.385,-0.617,-0.970
7,Sleep Pods,SLEEP_POD_POLYESTER,0.00115,643.6,0.5037,0.8879,0.519,-0.146,0.070,0.584


## 2. Extracción del Vector de Cointegración (Johansen)
Procedemos a **extraer el primer autovector** (eigenvector) asociado al mayor valor propio. Este vector nos proporciona el portafolio estacionario lineal óptimo dentro de la cesta entera.

In [7]:
johansen_portfolios = []

for cat, cols in CATEGORIES.items():
    cat_df = prices[cols].dropna()
    try:
        # det_order=0 asume no tendencia lineal; k_ar_diff=1 lags de diferencias
        res = coint_johansen(cat_df, det_order=0, k_ar_diff=1)
        
        # El autovector correspondiente al mayor autovalor es res.evec[:, 0]
        eigenvector = res.evec[:, 0]
        
        # Normalizamos el vector para que el primer activo tenga peso 1.0 
        # (esto facilita operar 1 unidad del activo 1 contra proporciones del resto)
        norm_vector = eigenvector / eigenvector[0]
        
        # Construir el spread del portafolio: S_t = w1*P1 + w2*P2 ... w5*P5
        portfolio_spread = (cat_df * norm_vector).sum(axis=1)
        
        adf_res = adf_test(portfolio_spread)
        hl = half_life(portfolio_spread)
        hu = hurst_exponent(portfolio_spread)
        
        johansen_portfolios.append({
            'category': cat,
            'adf_pval': round(adf_res['adf_pval'], 5),
            'half_life': round(hl, 1) if np.isfinite(hl) else 9999,
            'hurst': round(hu, 4) if not np.isnan(hu) else np.nan,
            'w1': round(norm_vector[0], 3),
            'w2': round(norm_vector[1], 3),
            'w3': round(norm_vector[2], 3),
            'w4': round(norm_vector[3], 3),
            'w5': round(norm_vector[4], 3)
        })
        
    except Exception as e:
        print(f"Error in Johansen for {cat}: {e}")

joh_port_df = pd.DataFrame(johansen_portfolios).sort_values('adf_pval')
print('\n=== Johansen Eigenvector Portfolios ===')
display(joh_port_df)


=== Johansen Eigenvector Portfolios ===


,category,adf_pval,half_life,hurst,w1,w2,w3,w4,w5
3,Pebbles,0.00000,0.7,0.0002,1.0,1.000,1.000,1.000,1.000
2,Microchips,0.00020,506.3,0.4982,1.0,-2.720,0.190,2.680,5.223
5,UV Visors,0.00082,599.7,0.4763,1.0,4.671,1.696,4.427,5.427
1,Sleep Pods,0.00120,684.6,0.5040,1.0,-1.161,-2.187,0.277,1.234
4,Robots,0.00194,390.5,0.4452,1.0,2.750,3.287,2.271,2.255
8,Shakes,0.00247,915.1,0.4640,1.0,1.379,0.407,-3.198,2.148
9,Snacks,0.00255,544.2,0.4805,1.0,0.722,-0.289,0.174,-0.117
6,Translators,0.00344,750.7,0.4928,1.0,1.182,-1.840,0.236,1.880
0,Galaxy Sounds,0.03938,1409.6,0.5128,1.0,0.062,-0.304,0.940,0.765
7,Panels,0.25426,2124.3,0.5156,1.0,0.324,1.192,1.382,0.802


## 3. Lógica de Ejecución bajo el Límite de 10 (Position Sizing)
Operar una cesta requiere una gestión de inventario meticulosa. 
Si tu modelo Johansen o Múltiple OLS dicta la ecuación: 
$$Target - 0.4 X_1 - 0.3 X_2 - 0.3 X_3 = 0$$

El algoritmo de *Netting* para operar una señal de `Z-Score > 2` sería:
1. Calcular el espacio máximo del Target: `max_qty_target = min(5, POS_LIMIT - current_pos)`.
2. Proyectar la necesidad de cobertura temporal:
   `hedge_X1 = round(-0.4 * max_qty_target)`
   `hedge_X2 = round(-0.3 * max_qty_target)`
3. Verificar que ninguna de las operaciones marginales cause un *Limit Breach* en su respectivo activo (`abs(pos + hedge) <= POS_LIMIT`).
4. Enviar todas las órdenes como un bloque simultáneo si los límites lo permiten.